# Mahabharata knowledge graph: Phase 4r, GLiREL relation extraction

Runs `src/phase4r_glirel.py` on a Kaggle GPU. GLiREL (`jackboyla/glirel-large-v0`) labels the relation between two names in a sentence, zero-shot, with labels such as *is the mother of*, *is the father of*, *is an ally of*, *is the charioteer of*. Its output is a **candidate layer**: nothing here is asserted until a threshold has been set on a hand-judged sample.

Inputs are two files you already have locally, and nothing else:
- `build/mentions.parquet` (Phase 1 names and sentences, about 5.7 MB)
- `build/coref.parquet` (Phase 4b coreference, about 0.7 MB), so "he" can stand for the person it was resolved to

## Before you run

1. **Make one private Kaggle dataset** with those two files. **Keep it and this notebook private**: the sentences are verbatim text of a copyrighted translation.
2. **Add Data**: attach the dataset.
3. **Settings → Accelerator → GPU T4 x1 or P100.**
4. **Settings → Internet → On**, for `pip install glirel` and the model download.
5. **Run a smoke test first**: Cell 5 has `LIMIT = 300`. Run all cells and read Cell 6: the first raw GLiREL output, the label table, and the top examples with their sentences. If the examples look sensible, set `LIMIT = 0` and use **Save Version → Save & Run All (Commit)** for the full run (about 26,000 sentences).

## Afterwards, locally

Download `mahabharata_glirel_outputs.zip` from the Output tab and unzip it into the repo root. It holds `build/glirel_relations.parquet`, `build/glirel_report.md` and `build/glirel_manifest.json`.

In [ ]:
# Cell 1: find the two inputs and set up a working copy.
import hashlib, os, shutil, sys
from pathlib import Path

INPUT = Path(os.environ.get("MBH_INPUT", "/kaggle/input"))
WORK = Path(os.environ.get("MBH_WORK", "/kaggle/working/mahabharata-main"))
(WORK / "build").mkdir(parents=True, exist_ok=True)
(WORK / "src").mkdir(parents=True, exist_ok=True)

def first(pattern):
    hits = sorted(INPUT.rglob(pattern), key=lambda p: len(p.parts))
    if not hits:
        raise SystemExit("No %s under %s. Attach the dataset holding it." % (pattern, INPUT))
    return hits[0]

for pattern, name in [("mentions*.parquet", "mentions.parquet"), ("coref*.parquet", "coref.parquet")]:
    p = first(pattern)
    shutil.copy2(p, WORK / "build" / name)
    print("%-17s <- %s  sha256 %s" % (name, p, hashlib.sha256(p.read_bytes()).hexdigest()[:16]))
os.chdir(WORK)   # the script reads and writes build/ relative to the repo root

In [ ]:
# Cell 2: dependencies. torch, pandas and spaCy ship with Kaggle.
# glirel's dependency seqeval exists only as a source archive, and building it
# fails on Kaggle's Python 3.13. glirel imports exactly one helper from it,
# _prf_divide, and uses it only to score training runs. So: try the normal
# install; if it fails, install glirel without dependencies, add what it
# imports, retry seqeval with build tools, and as a last resort write that one
# helper as a stand-in seqeval. Prediction never calls it.
import site, subprocess, textwrap

def pip(*args):
    return subprocess.run([sys.executable, "-m", "pip", "install", *args]).returncode

if pip("-q", "glirel") != 0:
    print("\nPlain install failed (see the error above). Installing glirel without seqeval.\n")
    if pip("-q", "--no-deps", "glirel") != 0:
        raise SystemExit("glirel itself would not install.")
    pip("-q", "loguru", "pyyaml", "datasets", "huggingface_hub", "tqdm")
    pip("-q", "setuptools", "wheel")
    if pip("-q", "--no-build-isolation", "seqeval") != 0:
        shim = Path(site.getsitepackages()[0]) / "seqeval" / "metrics"
        shim.mkdir(parents=True, exist_ok=True)
        (shim.parent / "__init__.py").write_text("")
        (shim / "__init__.py").write_text("")
        (shim / "v1.py").write_text(textwrap.dedent('''
            """Stand-in for seqeval.metrics.v1, written by the notebook because
            seqeval would not build here. glirel imports only this helper, and
            only uses it to score training runs. Same arithmetic as seqeval's."""
            import numpy as np

            def _prf_divide(numerator, denominator, metric, modifier, average,
                            warn_for, zero_division="warn"):
                mask = denominator == 0.0
                denominator = denominator.copy()
                denominator[mask] = 1
                result = numerator / denominator
                if np.any(mask):
                    result[mask] = 0.0 if zero_division in ("warn", 0) else 1.0
                return result
        '''))
        print("seqeval would not build; wrote the stand-in at", shim / "v1.py")

from importlib.metadata import version
for pkg in ("glirel", "torch", "transformers"):
    print(pkg, version(pkg))
from glirel import GLiREL  # noqa: F401  the import that failed before
print("glirel imports cleanly")

In [ ]:
# Cell 3: the GPU check. Fail here, not an hour in.
import torch
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Settings -> Accelerator -> GPU, then restart.")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
%%writefile /kaggle/working/mahabharata-main/src/phase4r_glirel.py
"""Phase 4r. Zero-shot relation extraction with GLiREL, over the whole corpus once.

The ten predicates of Phase 4 come from hand-written spaCy patterns, and the
graph has no mother or father, no ally, enemy or charioteer. GLiREL
(Boylan et al. 2025, jackboyla/glirel-large-v0) classifies the relation between
two given entity spans against labels written as plain text, with no training.
It does not find entities itself, so the spans come from what this project
already has: the Phase 1 name mentions and the Phase 4b coreference spans, which
let "he" stand for the person it was resolved to.

Output is a candidate layer, never asserted: build/glirel_relations.parquet,
one row per (sentence, head span, tail span) whose best label is a relation,
with its score. Scores are not probabilities. A threshold has to be set against
a hand-judged sample before any of this is used as fact.

Coreference runs before this and needs a GPU; so does this, in practice. On a
CPU it is a few sentences a second, so it runs on Kaggle.

Run: python src/phase4r_glirel.py [--limit N] [--dry-run] [--model NAME]
  --dry-run   build every input and check each span maps back to its text, with
              no model loaded. This is the part that can be verified anywhere.
"""
import argparse
import collections
import json
import os
import re
import time

import pandas as pd

MODEL = "jackboyla/glirel-large-v0"
OUT = "build/glirel_relations.parquet"
PARTS = "build/glirel_parts"
REPORT = "build/glirel_report.md"
CHECKPOINT = 2000          # sentences per saved part, so a lost session resumes
BATCH = 16                 # sentences per GPU call
MAX_TOKENS = 200           # longer sentences are skipped and counted, not cut
MAX_WIDTH = 12             # the model's max_width: a wider entity span cannot be scored
THRESHOLD = 0.10           # kept low on purpose: the cut-off is set later, on judged data
ENTITY_LABELS = ("PERSON", "GPE")

# Head -> label -> tail, phrased so the sentence "head label tail" reads true.
# "no relation" absorbs pairs that are merely named together.
LABELS = [
    "is the mother of", "is the father of", "is the son of", "is the daughter of",
    "is the wife of", "is the husband of", "is the brother of", "is the sister of",
    "is the grandson of", "is the teacher of", "is the charioteer of",
    "is an ally of", "is an enemy of", "is a friend of", "killed",
    "is the king of", "no relation",
]
# Labels whose tail must be a place; every other label is person to person.
PLACE_TAIL = {"is the king of"}

TOKEN = re.compile(r"[^\W_]+|[^\w\s]")


def tokenize(sentence):
    """Tokens for GLiREL and each token's character span.

    Words and punctuation are separate tokens, as spaCy writes them, so
    "Kunti’s" gives Kunti a token of its own and "Ayoda-Dhoumya" is two words.
    """
    found = list(TOKEN.finditer(sentence))
    return [m.group() for m in found], [(m.start(), m.end()) for m in found]


def token_span(offsets, start, end):
    """Character span -> (first token, last token) it covers, or None."""
    toks = [i for i, (a, b) in enumerate(offsets) if a < end and b > start]
    return (toks[0], toks[-1]) if toks else None


def find_name(tokens, surface, hint):
    """Token span of a name in the sentence, or None.

    Phase 1's tok_start counts words in the NER file's own word list, which
    splits hyphens and drops footnote numbers, so a third of them do not line
    up with the sentence text. The name is found by its words instead, and the
    stored position only chooses between repeated occurrences.
    """
    want = [w.lower() for w in TOKEN.findall(surface) if w[0].isalnum()]
    if not want:
        return None
    words = [i for i, t in enumerate(tokens) if t[0].isalnum()]
    low = [tokens[i].lower() for i in words]
    hits = [k for k in range(len(low) - len(want) + 1) if low[k:k + len(want)] == want]
    if not hits:
        return None
    k = min(hits, key=lambda k: abs(k - hint))
    return words[k], words[k + len(want) - 1]


def build_inputs(mentions, coref):
    """One record per sentence with at least two entity spans."""
    m = mentions[mentions.label.isin(ENTITY_LABELS)]
    sents = mentions.drop_duplicates(["parva", "sent_id"]).set_index(["parva", "sent_id"])
    co = {k: g for k, g in coref.groupby(["parva", "sent_id"])} if len(coref) else {}
    stats = collections.Counter()
    out = []
    keys = sorted(set(zip(m.parva, m.sent_id)) | set(co))
    by_sent = {k: g for k, g in m.groupby(["parva", "sent_id"])}
    for key in keys:
        if key not in sents.index:
            continue
        s = sents.loc[key]
        tokens, offsets = tokenize(str(s.sentence))
        if len(tokens) > MAX_TOKENS:
            stats["skipped_long"] += 1
            continue
        ents, used = [], set()
        for r in by_sent.get(key, pd.DataFrame()).itertuples():
            span = find_name(tokens, r.surface, r.tok_start)
            if span is None:
                stats["mention_not_found"] += 1
                continue
            if span[1] - span[0] + 1 > MAX_WIDTH:
                stats["mention_too_wide"] += 1
                continue
            if used & set(range(span[0], span[1] + 1)):
                continue          # the same words, already taken by another mention
            ents.append({"span": list(span), "type": r.label, "surface": r.surface,
                         "name": r.surface, "via": "mention"})
            used.update(range(span[0], span[1] + 1))
        for r in co.get(key, pd.DataFrame()).itertuples():
            if r.person != 3:
                continue          # I and you inside speech: a different problem, noted in 4b
            span = token_span(offsets, int(r.start), int(r.end))
            if span is None or used & set(range(span[0], span[1] + 1)):
                continue          # the span is already a named mention
            if span[1] - span[0] + 1 > MAX_WIDTH:
                stats["coref_too_wide"] += 1
                continue          # a long description; the model cannot score it
            ents.append({"span": list(span), "type": "PERSON",
                         "surface": r.surface, "name": r.canonical, "via": "coref"})
            used.update(range(span[0], span[1] + 1))
        if len(ents) < 2:
            continue
        stats["sentences"] += 1
        stats["entities"] += len(ents)
        stats["coref_entities"] += sum(e["via"] == "coref" for e in ents)
        out.append({"parva": key[0], "sent_id": int(key[1]),
                    "adhyaya_ordinal": s.adhyaya_ordinal, "cited": bool(s.chapter_cited),
                    "tokens": tokens, "entities": ents})
    return out, stats


def check_spans(records):
    """Every span must read back as its surface, mentions and coref alike."""
    bad = []
    for rec in records:
        for e in rec["entities"]:
            text = " ".join(rec["tokens"][e["span"][0]:e["span"][1] + 1])
            if norm(text) != norm(e["surface"]):
                bad.append((rec["parva"], rec["sent_id"], e["via"], e["surface"], text))
    return bad


def norm(s):
    return re.sub(r"[\W_]+", "", s).lower()


def pos_of(rel, side):
    """GLiREL's span for head or tail, as the inclusive [start, end] it was given.

    glirel 1.2.1 takes entity spans with an inclusive end and returns them with
    an exclusive one: model.py writes 'head_pos': [start, end + 1]. Looked up as
    returned, no relation matches any entity and every row is silently lost.
    """
    p = rel.get("%s_pos" % side)
    return None if p is None else [p[0], p[1] - 1]


def load_model(model_name, device):
    """GLiREL, loaded around a version clash in its own loader.

    glirel 1.2.1's _from_pretrained requires `proxies` and `resume_download`,
    which huggingface_hub 1.x no longer passes, so GLiREL.from_pretrained fails
    with a TypeError. The loader only forwards them to a download it skips when
    the files are already on disk. So the two files are fetched here with the
    current API, and the loader is called on that folder.
    """
    from glirel import GLiREL
    local = model_name
    if not os.path.isdir(model_name):
        from huggingface_hub import snapshot_download
        local = snapshot_download(model_name,
                                  allow_patterns=["pytorch_model.bin", "glirel_config.json"])
    return GLiREL._from_pretrained(model_id=local, revision=None, cache_dir=None,
                                   force_download=False, proxies=None, resume_download=False,
                                   local_files_only=True, token=None, map_location=device)


def run(records, model_name, done_parts):
    import torch
    model = load_model(model_name, "cuda" if torch.cuda.is_available() else "cpu")
    os.makedirs(PARTS, exist_ok=True)
    shown = False
    returned = kept = 0
    rows, t0, done_now = [], time.time(), 0
    todo = [i for i in range(len(records)) if i // CHECKPOINT not in done_parts]
    for b in range(0, len(todo), BATCH):
        idx = todo[b:b + BATCH]
        batch = [records[i] for i in idx]
        ners = [[[e["span"][0], e["span"][1], e["type"], e["surface"]] for e in rec["entities"]]
                for rec in batch]
        outs = model.batch_predict_relations([rec["tokens"] for rec in batch], LABELS,
                                             threshold=THRESHOLD, ner=ners, top_k=1)
        for i, rec, rels in zip(idx, batch, outs):
            new = extract(rec, rels)
            rows += new
            kept += len(new)
            returned += sum(r["label"] != "no relation" for r in rels)
            if not shown and rels:
                print("first raw GLiREL output:", rels[0], flush=True)
                shown = True
            # A position mismatch drops every relation without an error. If the
            # model has returned relations and none survived, stop and say so.
            if returned >= 50 and kept == 0:
                raise SystemExit("%d relations returned and none matched an entity span. "
                                 "Check pos_of() against this glirel version's output." % returned)
            done_now += 1
            if (i + 1) % CHECKPOINT == 0 or i == len(records) - 1:
                part = i // CHECKPOINT
                pd.DataFrame(rows).to_parquet("%s/part_%04d.parquet" % (PARTS, part))
                print("sentences %d/%d, part %d saved, %d relations in it, %.1f sentences/s"
                      % (i + 1, len(records), part, len(rows),
                         done_now / max(time.time() - t0, 1e-6)), flush=True)
                rows = []


def extract(rec, rels):
    """The kept rows for one sentence: a real label, both spans known, types right."""
    by_span = {tuple(e["span"]): e for e in rec["entities"]}
    rows = []
    for rel in rels:
        if rel["label"] == "no relation":
            continue
        h = by_span.get(tuple(pos_of(rel, "head") or ()))
        t = by_span.get(tuple(pos_of(rel, "tail") or ()))
        if h is None or t is None or h["name"] == t["name"]:
            continue
        want_place = rel["label"] in PLACE_TAIL
        if h["type"] != "PERSON" or (t["type"] == "GPE") != want_place:
            continue          # type constraints, applied after the model
        rows.append({"parva": rec["parva"], "sent_id": rec["sent_id"],
                     "adhyaya_ordinal": rec["adhyaya_ordinal"], "cited": rec["cited"],
                     "head": h["name"], "head_surface": h["surface"], "head_via": h["via"],
                     "head_span": h["span"], "label": rel["label"],
                     "tail": t["name"], "tail_surface": t["surface"], "tail_via": t["via"],
                     "tail_span": t["span"], "score": float(rel["score"])})
    return rows


def report(stats, frame, args):
    lines = ["# Phase 4r. GLiREL zero-shot relation extraction", "",
             "Generated by `src/phase4r_glirel.py`. Candidates only, never asserted. "
             "Scores are not probabilities; no threshold has been judged yet.", "",
             "- Model: `%s`, top label per pair, kept at score >= %.2f" % (args.model, THRESHOLD),
             "- Sentences with two or more entity spans: %d (%d skipped as longer than %d tokens)"
             % (stats["sentences"], stats["skipped_long"], MAX_TOKENS),
             "- Entity spans: %d, of which from coreference: %d"
             % (stats["entities"], stats["coref_entities"]),
             "- Relations kept: %d" % len(frame), ""]
    if len(frame):
        lines += ["| label | relations | score >= 0.5 | median score |", "|---|---|---|---|"]
        for lab, g in frame.groupby("label"):
            lines.append("| %s | %d | %d | %.2f |" % (lab, len(g), (g.score >= 0.5).sum(),
                                                     g.score.median()))
    open(REPORT, "w", encoding="utf-8").write("\n".join(lines) + "\n")
    print("\n".join(lines))


def main():
    ap = argparse.ArgumentParser(description=__doc__.split("\n")[0])
    ap.add_argument("--limit", type=int, default=0, help="first N sentences only: a smoke test")
    ap.add_argument("--dry-run", action="store_true", help="build inputs and check spans, no model")
    ap.add_argument("--model", default=MODEL)
    args = ap.parse_args()

    mentions = pd.read_parquet("build/mentions.parquet")
    coref = pd.read_parquet("build/coref.parquet") if os.path.exists("build/coref.parquet") \
        else pd.DataFrame()
    records, stats = build_inputs(mentions, coref)
    if args.limit:
        records = records[:args.limit]
    bad = check_spans(records)
    print("sentences %d, entity spans %d (coref %d), long skipped %d, names not found %d, "
          "spans wider than %d skipped %d, span mismatches %d"
          % (len(records), sum(len(r["entities"]) for r in records),
             sum(e["via"] == "coref" for r in records for e in r["entities"]),
             stats["skipped_long"], stats["mention_not_found"], MAX_WIDTH,
             stats["mention_too_wide"] + stats["coref_too_wide"], len(bad)), flush=True)
    for b in bad[:10]:
        print("  span mismatch:", b)
    if args.dry_run:
        return

    done = set()
    if os.path.isdir(PARTS) and not args.limit:
        done = {int(f[5:9]) for f in os.listdir(PARTS) if f.startswith("part_")}
        if done:
            print("resuming: parts %s already saved" % sorted(done), flush=True)
    run(records, args.model, done)
    parts = sorted(os.listdir(PARTS))
    frame = pd.concat([pd.read_parquet(os.path.join(PARTS, f)) for f in parts],
                      ignore_index=True) if parts else pd.DataFrame()
    frame.to_parquet(OUT)
    report(stats, frame, args)
    json.dump({"model": args.model, "limit": args.limit, "rows": len(frame)},
              open("build/glirel_manifest.json", "w"))


if __name__ == "__main__":
    main()

In [ ]:
# Cell 5: run. LIMIT > 0 is a smoke test on the first N sentences; 0 is the full corpus.
# A full run saves a part every 2,000 sentences in build/glirel_parts and resumes from them.
import subprocess, time
LIMIT = 300

shutil.rmtree("build/glirel_parts", ignore_errors=True) if LIMIT else None
cmd = [sys.executable, "src/phase4r_glirel.py"] + (["--limit", str(LIMIT)] if LIMIT else [])
print("====", " ".join(cmd), flush=True)
t = time.time()
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                     env=dict(os.environ, PYTHONUNBUFFERED="1"))
for line in p.stdout:
    print(line, end="", flush=True)
rc = p.wait()
print("\nexit %d after %.1f minutes" % (rc, (time.time() - t) / 60))
if rc:
    raise SystemExit("phase4r_glirel.py failed. Read the output above.")

In [ ]:
# Cell 6: read the results before trusting them: the label table, then the
# highest-scoring examples with their sentences, so a human can see whether the
# labels and their direction are right.
import pandas as pd
from IPython.display import Markdown, display
display(Markdown(Path("build/glirel_report.md").read_text(encoding="utf-8")))
rel = pd.read_parquet("build/glirel_relations.parquet")
sent = pd.read_parquet("build/mentions.parquet").drop_duplicates(["parva", "sent_id"]).set_index(["parva", "sent_id"]).sentence
pd.set_option("display.max_colwidth", 160)
for label, g in rel.sort_values("score", ascending=False).groupby("label", sort=False):
    print("\n==", label)
    for r in g.head(3).itertuples():
        print("  %.2f  %s -> %s  [%s %s]  %s" % (r.score, r.head, r.tail, r.parva, r.sent_id,
                                              sent.get((r.parva, r.sent_id), "")[:150]))

In [ ]:
# Cell 7: bundle for the Output tab. The zip unpacks into the repo root.
# The parquet holds no sentence text, only names, spans and scores, but keep it private anyway.
import zipfile
dest = Path("/kaggle/working/mahabharata_glirel_outputs.zip")
with zipfile.ZipFile(dest, "w", zipfile.ZIP_DEFLATED) as z:
    for rel_path in ["build/glirel_relations.parquet", "build/glirel_report.md", "build/glirel_manifest.json"]:
        z.write(WORK / rel_path, rel_path)
print(dest, "%.1f MB" % (dest.stat().st_size / 1e6))
if LIMIT:
    print("WARNING: smoke test of %d sentences. Set LIMIT = 0 for the full run." % LIMIT)